# 02 · Liquidity, fragmentation and market quality around the SVB failure

**Question.** When Silicon Valley Bank failed (Fri 10 Mar 2023) and Circle disclosed $3.3bn of USDC reserves stuck
there, did trading conditions deteriorate in USDC-quoted markets specifically, and did liquidity flee to USDT?

Regimes: pre-crisis 1–9 Mar · peak crisis 10–13 Mar · recovery 14–21 Mar.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import config, microstructure as ms, basis, plotting
from stablecoin_risk.data import loaders
from stablecoin_risk.pipeline import arbitrage_gap_panel

panel = loaders.load_svb_panel()   # gap-filled data, dead feeds removed
targets = [k for k in panel if k.rsplit("_", 1)[1] in config.BTC_PAIRS + config.STABLECOIN_PAIRS
           and k not in config.MIRRORED_MARKETS]
metrics = {k: ms.compute_all_metrics(panel[k]) for k in targets}
sorted(metrics)

['binance_BTCUSD',
 'binance_BTCUSDC',
 'binance_BTCUSDT',
 'binance_USDCUSD',
 'binance_USDTUSD',
 'coinbase_BTCUSD',
 'coinbase_BTCUSDT',
 'crypto_com_BTCUSD',
 'crypto_com_BTCUSDT']

## 1. Peg deviation

The most direct evidence: how far did each stablecoin trade from $1 on Binance.US?

In [3]:
depeg = {k: (ms.original_close(panel[k]) - 1).abs() * 1e4 for k in panel if k.endswith(("USDCUSD", "USDTUSD"))}
pd.DataFrame({k: {r: v.loc[a:f"{b} 23:59"].median() for r, (a, b) in config.REGIMES.items()} | {"max": v.max()}
              for k, v in depeg.items()}).T.rename_axis("median |P-1| in bps")

,pre_crisis,peak_crisis,recovery,max
median |P-1| in bps,,,,
binance_USDCUSD,1,207,5,"1,367"
binance_USDTUSD,1,47,27,165


In [4]:
fig = plotting.plot_svb_panels({"Peg deviation |P - 1| (60-min mean)": depeg}, "Basis points", resample="60min")

## 2. Crisis multipliers

Peak-crisis median ÷ pre-crisis median of each estimator. Spread estimators use 1,440-bar rolling windows and
price-impact measures 60-minute windows, all backward-looking.

In [5]:
regime = ms.regime_statistics(metrics)
ms.crisis_multipliers(regime).round(2)

,amihud_x,kyle_obizhaeva_x,roll_spread_x,cs_spread_x,ar_spread_x,vpin_x
market,,,,,,
binance_BTCUSD,0.76,1.31,0,2.06,0,1.18
binance_BTCUSDC,3.62,1.8,6.37,6.41,14.54,1.57
binance_BTCUSDT,0.63,1.23,0,2.09,0,1.24
binance_USDCUSD,11.85,3.86,5.06,162.5,6.32,1.69
binance_USDTUSD,1.28,1.24,1.95,3.05,2.68,1.95
coinbase_BTCUSD,0.8,1.26,0,2.07,NaN,1.17
coinbase_BTCUSDT,0.36,1.17,0,2.2,0,1.15
crypto_com_BTCUSD,3.24,1.8,0,1.87,0,0.74
crypto_com_BTCUSDT,0.54,1.44,0,2.55,NaN,1.55


USDC-quoted books deteriorate far more than USD- or USDT-quoted ones: Amihud price impact rises ~3.6× for BTC/USDC
and ~12× for USDC/USD, against <1.3× for BTC/USD and BTC/USDT. (The Roll estimator is zero for most BTC books
because one-minute price changes are positively autocorrelated; it is reported for completeness only.)

## 3. Cross-venue arbitrage gaps

$\text{Gap}_t = |P^{A}_t - P^{B}_t| / P^{B}_t \times 10^4$ on minutes where **both** venues printed a trade.
Stablecoin/USD on Coinbase is implied from the USDC/EUR and USDT/EUR books × EUR/USD (FX forward-filled over the
weekend). The original paper compared Binance with Kraken; the Kraken one-minute files are not part of this
repository, and the pipeline adds them automatically if placed in `data/raw/kraken/`.

In [6]:
btc_gaps, stable_gaps = arbitrage_gap_panel(panel)
gaps = pd.concat([btc_gaps, stable_gaps], axis=1)
pd.DataFrame({c: {r: gaps[c].loc[a:f"{b} 23:59"].median() for r, (a, b) in config.REGIMES.items()} | {"max": gaps[c].max()}
              for c in gaps}).T.round(1)

,pre_crisis,peak_crisis,recovery,max
BTCUSD binance vs coinbase,0.8,4.7,3.1,82.6
BTCUSD binance vs crypto_com,0.9,30.5,4,117.9
BTCUSDC binance vs coinbase (unified USD book),1.4,109.8,4.1,"1,425"
BTCUSDT binance vs coinbase,1.2,2.6,2.5,55.3
BTCUSDT binance vs crypto_com,1,2.1,1.8,44.7
USDCUSD binance vs coinbase (EUR-implied),10.6,70.8,20.2,"1,153"
USDTUSD binance vs coinbase (EUR-implied),9.5,57.4,18.1,248.1


In [7]:
fig = plotting.plot_svb_panels({"BTC pairs": dict(btc_gaps.items()), "Stablecoin / USD": dict(stable_gaps.items())},
                               "Basis points", resample="5min", legend_loc="upper right")

BTC/USD and BTC/USDT stay within a few tens of bps across venues throughout, while **BTC/USDC dislocates by more
than 1,000 bps** on the night of 10–11 March: arbitrage capital would not warehouse USDC while redemptions were
in doubt. The same pattern appears directly in USDC/USD.

## 4. Market Quality Index (MQI) and Stablecoin Peg Quality Index (SPQI)

$\text{MQI}_t = 100 \sum_k w_k (1 - s_{k,t})$ where each score $s_k \in [0,1]$ maps a metric's z-score versus the
pre-crisis baseline onto the unit interval: spread (Corwin–Schultz, 30%), price impact (Amihud, 30%),
efficiency stress (|ER − baseline|, 20%), VPIN (20%).

In [8]:
mqi = ms.compute_mqi_panel(metrics, lambda k: k.startswith("binance_BTC"))
spqi = ms.compute_mqi_panel(metrics, lambda k: k.endswith(("USDCUSD", "USDTUSD")))
pd.DataFrame({k: {r: s.loc[a:f"{b} 23:59"].median() for r, (a, b) in config.REGIMES.items()}
              for k, s in {**mqi, **spqi}.items()}).T.round(1)

,pre_crisis,peak_crisis,recovery
binance_BTCUSD,76.9,63.3,58
binance_BTCUSDC,76.9,53.5,65.6
binance_BTCUSDT,76.7,65.9,59.6
binance_USDCUSD,78.9,40.8,56.6
binance_USDTUSD,76.8,44.9,47.9


In [9]:
fig = plotting.plot_svb_panels({"BTC MQI by quote currency (Binance)": {k.rsplit("_", 1)[1]: v for k, v in mqi.items()},
                                "Stablecoin Peg Quality Index": spqi},
                               "Composite score (0-100)", resample="60min", ylim=(0, 100), legend_loc="lower left")

BTC/USDC market quality falls furthest (median 77 → 54) versus 63 for BTC/USD and 66 for BTC/USDT. Both
peg markets lose quality (SPQI 79 → 41 for USDC, 77 → 45 for USDT) because USDT/USD absorbed the flight-to-safety
flow, but the *price* outcomes differ sharply: the median peak-crisis deviation from $1 was 207 bps for USDC and
47 bps for USDT (section 1). USDT got busier and stayed at par.

## 5. Cross-quote basis and Markov regimes

$b_t = (P^{BTC/USD}_t - P^{BTC/Q}_t)/P^{BTC/Q}_t$ is the market-implied discount on stablecoin $Q$. A two-state
Markov-switching mean model (5-minute sampling) dates the stress regime without imposing event windows.

In [10]:
b = basis.basis_panel(panel)
basis.basis_summary(b, config.REGIMES).round(2)

n  mean_bps  std_bps  min_bps  max_bps
series                  regime                                                 
binance_BTC_USD_USDC    pre_crisis    6636     -0.14     2.96   -22.18    21.74
                        peak_crisis   3725    -256.1    304.4   -1,252    26.04
                        recovery      6907     -6.25     9.45   -116.1    36.22
binance_BTC_USD_USDT    pre_crisis   12427      0.12     2.13   -30.62    25.05
                        peak_crisis   5683     49.15    38.37    -6.16    165.9
                        recovery     11423     28.19    10.47    -18.5    75.65
coinbase_BTC_USD_USDT   pre_crisis    9559      0.22     2.75   -18.83     56.9
                        peak_crisis   5016     56.29    42.77   -16.41    194.1
                        recovery      8772     32.47    10.24   -31.66    84.89
crypto_com_BTC_USD_USDT pre_crisis   12924     -0.16     1.43    -8.89     5.52
                        peak_crisis   5759      84.4    56.76    -6.35    193.5
                        recovery     11519     33.36    10.39     9.09    71.82

In [11]:
fits = {c: basis.fit_markov_switching(b[c]) for c in b if b[c].notna().sum() > 500}
basis.regime_table({c: f for c, f in fits.items() if f is not None}).round(3)

,calm_mean_bps,stress_mean_bps,p_stay_calm,p_stay_stress,expected_stress_duration_bars,first_stress_time,share_in_stress
series,,,,,,,
binance_BTC_USD_USDC,-2.16,-290.3,0.999,0.994,155.4,2023-03-11 03:20:00+00:00,0.158
binance_BTC_USD_USDT,0.123,37.88,0.999,1,"2,016",2023-03-03 01:20:00+00:00,0.532
coinbase_BTC_USD_USDT,0.229,44.99,0.997,0.998,407.9,2023-03-01 19:40:00+00:00,0.514
crypto_com_BTC_USD_USDT,-0.211,54.57,1,1,"6,222",2023-03-10 21:30:00+00:00,0.529


On Binance the BTC/USD–BTC/USDC basis switches into a stress regime on **11 March ~03:20 UTC**, a few hours after
Circle disclosed its SVB exposure. The regime mean is about −290 bps: BTC cost ~3% more in USDC. For USDT the
two-state model mostly separates "before" from "after": every venue's USD–USDT basis moves to a persistent
**premium** of 30–50 bps from 10 March that has not reverted by 21 March (recovery mean +28 to +33 bps in the
regime table). That is a lasting re-pricing of USDT as the preferred dollar proxy, not a transient spike.